# Analyse Exploratoire des Données (EDA)

### Import des Modules et configuration de l'affichage

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

# Configuration de l'affichage du notebook
plt.rcParams["figure.figsize"] = (10, 5)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

### Chargement et description du dataset

In [ ]:
# Emplacement du dataset
DATA_DIR = Path.home() / "Desktop" / "Data Engineer" / "Projet 06"
FILE_NAME = "2016_Building_Energy_Benchmarking.csv"

PATH_CSV = DATA_DIR / FILE_NAME
# Vérification de l'existence du fichier
if not PATH_CSV.exists():
    raise FileNotFoundError(f"Fichier introuvable : {PATH_CSV}")

# Import Dataset
building_consumption = pd.read_csv(PATH_CSV, sep=None, engine="python", encoding="utf-8", na_values=["", "NA", "N/A", "null", "Null"])

print("Dataset chargé avec succès")
print(f"Dimensions : {building_consumption.shape}")

In [ ]:
def infos_dataframe(mydf, df_name="DataFrame", use_display=False):
    
    print(f"\n===== {df_name} : Aperçu =====")
    if use_display:
        display(mydf.head())
    else:
        print(mydf.head())

    print(f"\n===== {df_name} : Dimensions =====")
    print("Le dataset contient ", mydf.shape[0], " lignes et ", mydf.shape[1], " colonnes.")

    print(f"\n===== {df_name} : Doublons =====")
    print("Il y a ", mydf.duplicated().sum(), " lignes en double.")
    if "OSEBuildingID" in mydf.columns:
        print("Doublons sur OSEBuildingID :", mydf["OSEBuildingID"].duplicated().sum())

    print(f"\n\n===== {df_name} : Info =====")
    mydf.info()

    print(f"\n\n\n\n===== {df_name} : Valeurs manquantes =====")
    missing = mydf.isna().sum()
    missing_pct = (missing / len(mydf) * 100).round(2)
    missing_df = pd.DataFrame({
        "missing_count": missing,
        "missing_pct": missing_pct
    }).sort_values("missing_pct", ascending=False)
    if use_display:
        display(missing_df[missing_df["missing_count"] > 0])
    else:
        print(missing_df[missing_df["missing_count"] > 0])
    
    print(f"\n\n\n\n===== {df_name} — Statistiques numériques =====")
    if use_display:
        display(mydf.describe())
    else:
        print(mydf.describe())

    print(f"\n\n\n\n===== {df_name} : Statistiques catégorielles =====")
    if mydf.select_dtypes(include="object").shape[1] > 0:
        if use_display:
            display(mydf.describe(include=["object"]))
        else:
            print(mydf.describe(include=["object"]))
    else:
        print("Aucune variable catégorielle.")

In [ ]:
# Affiche les informations du dataframe building_consumption
infos_dataframe(building_consumption, df_name="Dataset brut - Consommation énergétique", use_display=True)

### Analyse des targets (variables cibles)

In [ ]:
# Variables cibles du problème de régression
# (énergie et émissions à prédire.  Ne pas utiliser comme features)
TARGET_ENERGY = "SiteEnergyUse(kBtu)"
TARGET_GHG = "TotalGHGEmissions"

TARGETS = [TARGET_ENERGY, TARGET_GHG]

for target in TARGETS:
    if target not in building_consumption.columns:
        raise KeyError(f"Colonne cible absente dans df_filtered_pt : {target}")

In [ ]:
# Analyse de la complétude des variables cibles sélectionnées
print("Analyse de la complétude des variables cibles sélectionnées :\n")

n_total = len(building_consumption)

for target in TARGETS:
    df_temp_target = building_consumption[target]

    # Valeurs manquantes
    n_missing = df_temp_target.isna().sum()
    pct_missing = (n_missing / n_total) * 100

    # Valeurs à 0
    n_zero = (df_temp_target == 0).sum()
    pct_zero = (n_zero / n_total) * 100
    
    print(
        f"{target}\n"
        f" — valeurs manquantes : {n_missing} / {n_total} ({pct_missing:.2f}%)\n"
        f" - valeurs = 0 : {n_zero} / {n_total} ({pct_zero:.2f}%)\n"
    )

In [ ]:
# Conserve uniquement les lignes où au moins une des valeurs pour les targets sont présentes
n_before = len(building_consumption)

# Règle 1 : énergie obligatoire
mask_energy_valid = (building_consumption[TARGET_ENERGY].notna() & (building_consumption[TARGET_ENERGY] > 0))

# Règle 2 : émission C02 >= 0
mask_ghg_valid = building_consumption[TARGET_GHG].notna()

# Lignes exploitables
mask_valid_targets = mask_energy_valid & mask_ghg_valid

df_target_ok = building_consumption.loc[mask_valid_targets].copy()

if df_target_ok.empty:
    raise ValueError("Dataset vide après suppression des lignes sans target exploitable.")
    
n_after = len(df_target_ok)

print(f"Lignes supprimées (valeurs manquantes sur targets) : {n_before - n_after}")
print(f"Étape : filtrage des targets ({n_after} / {n_before})")

with pd.option_context("display.float_format", "{:.4f}".format):
    display(df_target_ok[TARGETS].isna().mean().to_frame("pct_missing"))
with pd.option_context("display.float_format", "{:.4f}".format):
    display((df_target_ok[TARGETS] == 0).mean().to_frame("pct_zero"))

print("Valeurs manquantes après filtrage : ")
display(df_target_ok[TARGETS].isna().sum().to_frame("n_missing"))

print("Valeurs à 0 après filtrage : ")
display((df_target_ok[TARGETS] == 0).sum().to_frame("n_zero"))

In [ ]:
# Colonnes connues pour provoquer une fuite de données (soit des données directes, soit des dérivées, soit des indicateurs basés sur les targets
COLS_LEAKAGE = [
    "SiteEnergyUseWN(kBtu)",
    "SteamUse(kBtu)",
    "Electricity(kWh)", "Electricity(kBtu)",
    "NaturalGas(therms)", "NaturalGas(kBtu)",
    "SiteEUI(kBtu/sf)", "SiteEUIWN(kBtu/sf)",
    "SourceEUI(kBtu/sf)", "SourceEUIWN(kBtu/sf)",
    "GHGEmissionsIntensity"
]

# Contrôle la présence des colonnes dans le dataset courant (il ne faut justement pas quelles soient présentes)
COLS_LEAKAGE_PRESENT = [c for c in COLS_LEAKAGE if c in df_target_ok.columns]

print(f"Colonnes de fuite identifiées dans le dataset : {len(COLS_LEAKAGE_PRESENT)}")
print(COLS_LEAKAGE_PRESENT)

# Vérifie que les targets ne sont pas utilisées comme features
overlap_with_targets = set(COLS_LEAKAGE_PRESENT) & set(TARGETS)
if overlap_with_targets:
    print(f"Colonnes de fuite recouvrant des targets : {list(overlap_with_targets)}")

### Valeurs manquantes et anomalies

In [ ]:
# Supprime la colonne Comments si elle ne contient ancune information exploitable
df_nocomment = df_target_ok.copy()

if "Comments" in df_nocomment.columns:
    comments_series = df_nocomment["Comments"]

    is_fully_empty = (
        comments_series.isna().all()
        or comments_series.dropna().astype(str).str.strip().eq("").all()
    )

    if is_fully_empty:
        df_nocomment.drop(columns=["Comments"], inplace=True)
        print("Colonne 'Comments' supprimée (entièrement vide)")
        print(f"Colonne 'Comments' supprimée : {df_nocomment.shape[1]} colonnes restantes")
    else:
        print("Colonne 'Comments' conservée (contient des informations)")

In [ ]:
# Teste si la colonne BuildingType est bien présente dans le dataframe
if "BuildingType" not in df_nocomment.columns:
    raise KeyError("Colonne 'BuildingType' absente")

n_before = len(df_nocomment)
missing_bt_pct = df_nocomment["BuildingType"].isna().mean()

df_bt = df_nocomment["BuildingType"].fillna("").astype(str).str.strip()

# On filtre les bâtiments par type de bâtiment pour ne garder que les non résidentiels
pattern = r"^(?:NonResidential|Campus|SPS)"
mask_non_res = df_bt.str.contains(pattern, case=False, regex=True)

df_filtered_bt = df_nocomment.loc[mask_non_res].copy()

# Informations du df après sélection des colonnes
print(f"Bâtiments non résidentiels conservés : {len(df_filtered_bt)} / {n_before} "
      f"({len(df_filtered_bt)/n_before:.1%})")
print(f"BuildingType manquants (avant filtre) : {missing_bt_pct:.1%}")
print("Répartition BuildingType conservés :")
print(df_filtered_bt["BuildingType"].value_counts(dropna=False).head(10))

In [ ]:
# Teste si la colonne PrimaryPropertyType est présente dans le dataframe
if "PrimaryPropertyType" not in df_filtered_bt.columns:
    raise KeyError("Colonne 'PrimaryPropertyType' absente")

# Filtre également sur le type d'utilisation des bâtiments
EXCLUDED_PRIMARY_PROPERTY_TYPES  = [
    "High-Rise Multifamily",
    "Low-Rise Multifamily",
    "Mid-Rise Multifamily",
    "Residence Hall",
    "Senior Care Community",
    "Mixed Use Property"
]

n_before = len(df_filtered_bt)

mask_non_residential_use = ~df_filtered_bt["PrimaryPropertyType"].isin(EXCLUDED_PRIMARY_PROPERTY_TYPES)
df_filtered_pt = df_filtered_bt.loc[mask_non_residential_use].copy()

n_after = len(df_filtered_pt)

# Informations du df après sélection des colonnes
print(f"Bâtiments non résidentiels conservés : {len(df_filtered_pt)} / {n_before} "
      f"({len(df_filtered_pt)/n_before:.1%})")
print("Répartition BuildingType conservés :")
print(df_filtered_pt["BuildingType"].value_counts(dropna=False).head(10))

In [ ]:
# Vérification de la cohérence des surfaces

# Colonnes des surfaces à contrôler
REQUIRED_GFA_COLS = [
    "PropertyGFABuilding(s)",
    "PropertyGFAParking",
    "PropertyGFATotal"
]

# Vérification de la présence des colonnes choisies
for col in REQUIRED_GFA_COLS:
    if col not in df_filtered_pt.columns:
        raise KeyError(f"Colonne GFA absente : {col}")

# Seuil de tolérance (écart accepté de 5%)
GFA_DIFF_THRESHOLD = 0.05 

# Copie du dataset
df_gfa_check = df_filtered_pt.copy()

# Calcul de la somme des surface des bâtiments et parking
df_gfa_check["GFA_sum_components"] = (
    df_gfa_check["PropertyGFABuilding(s)"].fillna(0) + df_gfa_check["PropertyGFAParking"].fillna(0)
)

# Calcul de l'écart relatif entre surface totale et somme des composants
df_gfa_check["GFA_diff_ratio"] = np.where(
    df_gfa_check["PropertyGFATotal"] > 0,
    (df_gfa_check["PropertyGFATotal"] - df_gfa_check["GFA_sum_components"]).abs()
    / df_gfa_check["PropertyGFATotal"],
    np.nan
)

# Bâtiments dont l'écart dépasse le seuil toléré
df_gfa_check["GFA_incoherent"] = df_gfa_check["GFA_diff_ratio"] > GFA_DIFF_THRESHOLD

# Proportion de bâtiments présentant une incohérence de surface
incoherent_ratio = df_gfa_check["GFA_incoherent"].mean()
print(f"Proportion de bâtiments avec incohérence GFA > "
      f"{GFA_DIFF_THRESHOLD:.0%} : {incoherent_ratio:.3f}"
)

In [ ]:
# Harmonisation de la variable d'usage principal du bâtiment

# Copie du dataset
df_larg_prop_use_type = df_gfa_check.copy()

# Vérification de la présence des colonnes nécessaires
if "LargestPropertyUseType" in df_larg_prop_use_type.columns and "PrimaryPropertyType" in df_larg_prop_use_type.columns:
    
    # Nombre de valeurs manquantes avant harmonisation
    n_missing_before = df_larg_prop_use_type["LargestPropertyUseType"].isna().sum()

    # Complète LargestPropertyUseType avec la valeur de PrimaryPropertyType lorsqu'il est manquant
    df_larg_prop_use_type["LargestPropertyUseType"] = (
        df_larg_prop_use_type["LargestPropertyUseType"].fillna(df_larg_prop_use_type["PrimaryPropertyType"])
    )

    # Nombre de valeurs manquantes après harmonisation
    n_missing_after = df_larg_prop_use_type["LargestPropertyUseType"].isna().sum()
    
    print(
        f"Harmonisation LargestPropertyUseType : "
        f"{n_missing_before - n_missing_after} valeurs complétées "
        f"({n_missing_after} restantes)"
    )
    pct_filled = (n_missing_before - n_missing_after) / max(n_missing_before, 1)
    print(f"Soit {pct_filled:.1%} des valeurs manquantes corrigées")
else:
    raise KeyError("Colonnes d'usage absentes pour l'harmonisation")

In [ ]:
# Préparation pour nettoyage structurel
REQUIRED_STRUCT_COLS = [
    "PropertyGFATotal",
    "LargestPropertyUseTypeGFA",
    "NumberofBuildings",
    "NumberofFloors"
]

for col in REQUIRED_STRUCT_COLS:
    if col not in df_larg_prop_use_type.columns:
        raise KeyError(f"Colonne absente pour nettoyage structurel : {col}")

# Df avant traitement
df_clean = df_larg_prop_use_type.copy()
n0 = len(df_clean)

# Surface totale strictement positive sinon on ne les garde pas
mask_valid_gfa = df_clean["PropertyGFATotal"].notna() & (df_clean["PropertyGFATotal"] > 0)
n_invalid_gfa = (~mask_valid_gfa).sum()
df_clean = df_clean[mask_valid_gfa]

# Surface Totale normalement supérieure à la surface d'usage
mask_incoherent_usage_gfa = (df_clean["LargestPropertyUseTypeGFA"].notna() &
                             df_clean["PropertyGFATotal"].notna() &
                            (df_clean["LargestPropertyUseTypeGFA"] > df_clean["PropertyGFATotal"]))
n_incoherent_usage = mask_incoherent_usage_gfa.sum()
df_clean = df_clean[~mask_incoherent_usage_gfa]
                             

# Bâtiments / étages incohérents → mise à NaN
# Minimum 1 bâtiment pour exister (doit > 0)
df_clean.loc[df_clean["NumberofBuildings"].notna() & (df_clean["NumberofBuildings"] <= 0), "NumberofBuildings"] = np.nan
# 0 étage = rdc, donc seules les valeurs négatives sont incohérentes.
df_clean.loc[df_clean["NumberofFloors"].notna() & (df_clean["NumberofFloors"] < 0), "NumberofFloors"] = np.nan

# Résultats des traitements
print(f"Lignes supprimées (surface totale invalide) : {n_invalid_gfa}")
print(f"Lignes supprimées (surface d'usage incohérent) : {n_incoherent_usage}")
print(f"Lignes supprimées au total : {n0 - len(df_clean)}")

# Informations du df après sélection des colonnes
print(f"Bâtiments non résidentiels conservés : {len(df_clean)} / {n0} "
      f"({len(df_clean)/n0:.1%})")
print("Répartition BuildingType conservés :")
print(df_clean["BuildingType"].value_counts(dropna=False).head(10))

### Analyse

In [ ]:
# Analyse de la distribution des targets

# Quantiles utilisées pour cadrer les histogrammes
VAL_LOWER_Q = 0.01
VAL_UPPER_Q = 0.99

for target in TARGETS:
    # Suppression des valeurs manquantes
    data = df_clean[target].dropna()

    # Calcul des bornes avec les quantiles choisis
    q_low, q_high = data.quantile([VAL_LOWER_Q, VAL_UPPER_Q])
    # Borne les valeurs pour l'affichage (les données originales ne sont pas modifiées)
    data_clip = data.clip(q_low, q_high)

    # Histogramme cadré sur les quantiles pour une meilleure lisibilité
    plt.figure()
    sns.histplot(data_clip, bins=50)
    plt.title(f"Distribution de {target} ({VAL_LOWER_Q:.0%}–{VAL_UPPER_Q:.0%})")
    plt.xlabel(target)
    plt.ylabel("Nombre de bâtiments")
    plt.show()
    
    # Boxplot basé sur l'ensemble des valeurs.
    plt.figure()
    sns.boxplot(x=data)
    plt.title(f"Boxplot de {target} (valeurs complètes)")
    plt.xlabel(target)
    plt.show()

In [ ]:
# Relation entre surface totale GFA et targets

# Quantile utilisé pour le cadrage visuel (on garde 99 % des données)
SCATTER_VAL_Q = 0.99

# Quantile surface calculé une seule fois
q_surface = df_clean["PropertyGFATotal"].quantile(SCATTER_VAL_Q)

for target in TARGETS:

    # Quantile de la target pour limiter l'impact visuel des valeurs extrêmes
    q_target = df_clean[target].quantile(SCATTER_VAL_Q)

    df_plot = df_clean[
        (df_clean["PropertyGFATotal"] <= q_surface) &
        (df_clean[target] <= q_target)
    ]

    # Nuage de points
    plt.figure()
    sns.scatterplot(
        data=df_plot,
        x="PropertyGFATotal",
        y=target,
        alpha=0.4
    )

    plt.xscale("log")
    plt.yscale("log")

    plt.xlabel("Surface totale (GFA) — échelle log")
    plt.ylabel(f"{target} — échelle log")
    plt.title(f"Relation Surface vs {target}")

    plt.tight_layout()
    plt.show()

In [ ]:
# Analyse des targets par usage principal du bâtiment

USAGE_COL = "LargestPropertyUseType"
# Limite le nombre des catégories pour garder des graphiques lisibles
TOP_N_USAGES = 30

# Vérification de la présence de la colonne LargestPropertyUseType
if USAGE_COL not in df_clean.columns:
    raise KeyError(f"Colonne absente pour analyse par usage : {USAGE_COL}")

# Sélection des usages les plus représentés dans le dataset
top_usages = (
    df_clean[USAGE_COL]
    .value_counts()
    .head(TOP_N_USAGES)
    .index
)

df_usage = df_clean[df_clean[USAGE_COL].isin(top_usages)]

# Visualisation des distributions des targets par usage
for target in TARGETS:
    plt.figure()
    sns.boxplot(
        data=df_usage,
        x=USAGE_COL,
        y=target
    )
    plt.xticks(rotation=30, ha="right")
    plt.title(f"{target} par usage principal (top {TOP_N_USAGES})")
    plt.xlabel("Usage principal du bâtiment")
    plt.ylabel(target)
    plt.tight_layout()
    plt.show()

In [ ]:
# Analyse des targets par quartier

NEIGHBORHOOD_COL = "Neighborhood"
# Seuil minimal de bâtiments par quartier pour améliorer l'analyse.
MIN_BUILDINGS_PER_HOOD = 50

# Vérification de la présence de la colonne Neighborhood
if NEIGHBORHOOD_COL not in df_clean.columns:
    raise KeyError(f"Colonne absente pour analyse par quartier : {NEIGHBORHOOD_COL}")

# Sélection des quartiers suffisamment représentés
top_hoods = (
    df_clean[NEIGHBORHOOD_COL]
    .value_counts()
    .loc[lambda x: x >= MIN_BUILDINGS_PER_HOOD]
    .index
)

df_hoods = df_clean[df_clean[NEIGHBORHOOD_COL].isin(top_hoods)].copy()

print(f"Nombre de quartiers analysés : {df_hoods[NEIGHBORHOOD_COL].nunique()}")

# Visualisation des distributions des targets par quartier
for target in TARGETS:
    plt.figure()
    sns.boxplot(
        data=df_hoods,
        x=NEIGHBORHOOD_COL,
        y=target
    )
    plt.xticks(rotation=45, ha="right")
    plt.title(f"{target} par quartier (quartier ≥ {MIN_BUILDINGS_PER_HOOD} bâtiments)")
    plt.tight_layout()
    plt.show()

### Export du dataset final

In [ ]:
# Dataset EDA final
df_eda = df_clean.copy()

# Complète LargestPropertyUseType par PrimaryPropertyType s'il est manquant
df_eda["LargestPropertyUseType"] = df_eda["LargestPropertyUseType"].fillna(df_eda["PrimaryPropertyType"])

# Liste des colonnes à garder 
EDA_COLS_KEEP = [
    "OSEBuildingID",
    "BuildingType",
    "PrimaryPropertyType",
    "LargestPropertyUseType",
    "LargestPropertyUseTypeGFA",
    "Neighborhood",
    "YearBuilt",
    "NumberofBuildings",
    "NumberofFloors",
    "PropertyGFATotal",
    "PropertyGFABuilding(s)",
    "PropertyGFAParking",
    "Latitude",
    "Longitude",
    "TotalGHGEmissions",
    "SiteEnergyUse(kBtu)"
]

# Vérification de la présence des colonnes choisies
EDA_COLS_KEEP = [c for c in EDA_COLS_KEEP if c in df_eda.columns]
df_eda = df_eda[EDA_COLS_KEEP].copy()

# Vérification que le dataset n'est pas vide
if df_eda.empty:
    raise ValueError("Le dataset EDA final est vide - export annulé")

# Export du dataset nettoyé pour la phase de modélisation
OUTPUT_PATH = DATA_DIR  / "df_eda_clean.csv"
df_eda.to_csv(OUTPUT_PATH, index=False)

print("Export EDA terminé")
print(f"Dimensions finales : {df_eda.shape}")
print(f"Fichier exporté : {OUTPUT_PATH}")